# Spotify Track Promotion Strategy — Data Cleaning

**Student:** Aarav Vanamareddy  
**Project:** DSO 576 Music Project  
**Branch:** aarav-vanamareddy

## Project Vision

The goal of this project is to support track promotion strategy by helping a music marketing decision-maker identify which tracks may deserve greater promotional attention.

The cleaned dataset will allow tracks to be compared with historically successful tracks using track popularity as the primary performance measure and streaming information as supporting evidence. Comparisons will consider genre, release information, and audio characteristics such as danceability, energy, valence, acousticness, instrumentalness, speechiness, liveness, loudness, and tempo.

The analysis is intended to support promotion decisions rather than guarantee that a track will become a hit.

## Business Question

Which track characteristics are associated with stronger historical track performance, and how can those patterns help prioritize tracks for promotion?

**Primary metric:** `track_popularity`  
**Secondary performance evidence:** `streams_total`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("spotify_full.csv")

print("Pandas version:", pd.__version__)
print("Data file:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

Pandas version: 3.0.5
Data file: spotify_full.csv
File exists: True


In [2]:
df_original = pd.read_csv(DATA_PATH, low_memory=False)

print("Original rows:", len(df_original))
print("Original columns:", df_original.shape[1])
print("Original shape:", df_original.shape)

df_original.head()

Original rows: 1182354
Original columns: 83
Original shape: (1182354, 83)


,track_id,spotify_id,title,title_normalized,track_artist,album_id,album_name,release_year,release_date,duration_ms,...,mean_energy_by_decade,mean_valence_by_decade,mean_tempo_by_decade,mean_loudness_by_decade,acousticness_decline_idx,danceability_trend,decade_track_count,anomaly_flags,has_anomaly,release_year_precision_aug
0,spotify:track:0yLdNVWF3Srea0uzk55zFn,0yLdNVWF3Srea0uzk55zFn,Flowers,flowers,Miley Cyrus,7I0tjwFtxUwBC1vgyeMAax,Flowers,2023,NaN,200455,...,0.632695,0.426118,121.802360,-9.080332,0.323613,0.549115,201603,NaN,False,False
1,spotify:track:0sf12qNH5qcw8qpgymFOqD,0sf12qNH5qcw8qpgymFOqD,Blinding Lights,blinding lights,The Weeknd,2ZfHkwHuoAZrlz7RMj0PDz,Blinding Lights,2019,2019-11-29,201573,...,0.651554,0.440806,121.858284,-8.588164,0.305663,0.542672,530477,NaN,False,False
2,spotify:track:21jGcNKet2qwijlDFuPiPb,21jGcNKet2qwijlDFuPiPb,Circles,circles,Post Malone,4g1ZRSobMefqF6nelkgibi,Hollywood's Bleeding,2019,2019-09-06,215280,...,0.651554,0.440806,121.858284,-8.588164,0.305663,0.542672,530477,NaN,False,False
3,spotify:track:2b8fOow8UzyDFAE27YhOZM,2b8fOow8UzyDFAE27YhOZM,Memories,memories,Maroon 5,3nR9B40hYLKLcR0Eph3Goc,Memories,2019,2019-09-20,189486,...,0.651554,0.440806,121.858284,-8.588164,0.305663,0.542672,530477,NaN,False,False
4,spotify:track:7k4t7uLgtOxPwTpFmtJNTY,7k4t7uLgtOxPwTpFmtJNTY,Tusa,tusa,KAROL G,7mKevNHhVnZER3BLgI8O4F,Tusa,2019,2019-11-07,200960,...,0.651554,0.440806,121.858284,-8.588164,0.305663,0.542672,530477,NaN,False,False


## Source Preservation

The original `spotify_full.csv` file is loaded into `df_original` and is never overwritten by the cleaning process.

All transformations are applied to a separate DataFrame. The original CSV remains unchanged so that the cleaning process can always be reproduced from the source data.

In [3]:
print("=== ORIGINAL DATASET ===")
print(f"Rows: {df_original.shape[0]:,}")
print(f"Columns: {df_original.shape[1]:,}")

print("\n=== COLUMN NAMES ===")
print(df_original.columns.tolist())

print("\n=== DATA TYPES ===")
print(df_original.dtypes.to_string())

print("\n=== MISSING VALUES ===")
missing_original = (
    df_original.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_original = pd.DataFrame({
    "missing_count": missing_original,
    "missing_percent": (
        missing_original / len(df_original) * 100
    ).round(2)
})

display(missing_original[missing_original["missing_count"] > 0])

=== ORIGINAL DATASET ===
Rows: 1,182,354
Columns: 83

=== COLUMN NAMES ===
['track_id', 'spotify_id', 'title', 'title_normalized', 'track_artist', 'album_id', 'album_name', 'release_year', 'release_date', 'duration_ms', 'explicit_flag', 'has_audio_features', 'source_dataset', 'energy', 'valence', 'danceability', 'acousticness', 'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo', 'key', 'key_name', 'mode', 'mode_name', 'time_signature', 'genre_l1', 'genre_l2', 'genre_raw', 'track_popularity', 'streams_total', 'streams_source', 'streams_total_estimated', 'stream_density', 'chart_peak_position', 'weeks_on_chart', 'chart_date_first', 'popularity_imputed_aug', 'track_age_days', 'era', 'decade', 'energy_loudness_ratio', 'mood_index', 'acoustic_synthetic_ratio', 'emotional_arc', 'tempo_band', 'is_likely_instrumental', 'is_likely_live', 'is_likely_spoken_word', 'loudness_war_index', 'novelty_score', 'hit_probability', 'mood_cluster', 'taste_cluster', 'fp_energy', 'fp_valence', 

,missing_count,missing_percent
taste_cluster,1180175,99.82
streams_source,1172754,99.19
weeks_on_chart,1167441,98.74
chart_date_first,1167441,98.74
chart_peak_position,1167441,98.74
anomaly_flags,1160967,98.19
release_date,1159764,98.09
album_name,1156965,97.85
album_id,1156961,97.85
title_normalized,25900,2.19


In [4]:
print("Total rows:", f"{len(df_original):,}")

for col in ["track_id", "spotify_id"]:
    print(f"\n{col}")
    print("Unique non-missing values:", f"{df_original[col].nunique(dropna=True):,}")
    print("Missing values:", f"{df_original[col].isna().sum():,}")
    print("Repeated non-missing IDs:", f"{df_original.loc[df_original[col].notna(), col].duplicated().sum():,}")

print("\nExact duplicate rows:",
      f"{df_original.duplicated().sum():,}")

Total rows: 1,182,354

track_id
Unique non-missing values: 1,182,354
Missing values: 0
Repeated non-missing IDs: 0

spotify_id
Unique non-missing values: 1,182,354
Missing values: 0
Repeated non-missing IDs: 0

Exact duplicate rows: 0


## Row Definition

One row in the analytical dataset is intended to represent one track.

`spotify_id` and `track_id` are investigated rather than automatically assumed to be unique. Exact duplicate rows and repeated identifiers are checked separately because repeated IDs may represent duplicate observations of the same track even when the rows are not exactly identical.

For the final analytical dataset, `spotify_id` is preferred as the track identifier when available, with `track_id` used as a fallback.

In [5]:
project_columns = [
    # Track identification
    "track_id",
    "spotify_id",
    "title",
    "track_artist",

    # Release / comparison groups
    "release_year",
    "release_date",
    "genre_l1",
    "genre_l2",

    # Track characteristics
    "duration_ms",
    "explicit_flag",
    "energy",
    "valence",
    "danceability",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness",
    "loudness",
    "tempo",
    "key",
    "mode",
    "time_signature",

    # Historical performance
    "track_popularity",
    "streams_total",
    "streams_source",
    "streams_total_estimated"
]

missing_expected_columns = [
    col for col in project_columns
    if col not in df_original.columns
]

print("Expected columns not found:", missing_expected_columns)

if missing_expected_columns:
    raise ValueError(
        f"Required columns are missing: {missing_expected_columns}"
    )

df = df_original[project_columns].copy()

print("\nOriginal shape:", df_original.shape)
print("Selected analytical shape:", df.shape)

Expected columns not found: []

Original shape: (1182354, 83)
Selected analytical shape: (1182354, 26)


## Column Selection Rule

The original dataset contains many variables that are not required for the Track Promotion Strategy business question, including downstream engineered scores and artist/decade aggregates.

The analytical dataset retains variables needed to:

1. identify tracks,
2. define relevant comparison groups such as genre and release year,
3. describe observable audio characteristics, and
4. measure historical track performance.

Precomputed downstream variables such as `hit_probability`, `mood_cluster`, `taste_cluster`, `sonic_evolution_score`, `longevity_score`, and `hit_rate` are not used in the cleaned analytical dataset. Excluding these variables keeps the analysis focused on interpretable track characteristics and avoids relying on unknown or downstream modeling assumptions.

The original source remains unchanged.

In [6]:
cleaning_decisions = pd.DataFrame([
    {
        "rule": "Keep only project-relevant analytical columns",
        "reason": "Focus the dataset on variables needed for track promotion decisions.",
        "check": "Confirm retained columns match the documented project column list."
    },
    {
        "rule": "Preserve the original source and clean a copy",
        "reason": "The source must remain reproducible and unchanged.",
        "check": "Compare df_original dimensions before and after cleaning."
    },
    {
        "rule": "Strip accidental leading/trailing whitespace from descriptive text",
        "reason": "Whitespace can create false category differences.",
        "check": "Count values changed by whitespace removal."
    },
    {
        "rule": "Do not alter track IDs",
        "reason": "Identifiers must preserve their original characters and leading zeros.",
        "check": "Compare IDs with the original data."
    },
    {
        "rule": "Convert release_date to a date type",
        "reason": "Dates should have a consistent type for time-based analysis.",
        "check": "Count failed date conversions."
    },
    {
        "rule": "Remove exact duplicate rows",
        "reason": "Identical observations would double-count the same information.",
        "check": "Verify no exact duplicates remain."
    },
    {
        "rule": "Use one record per track identifier",
        "reason": "The analytical grain is one track and repeated records could overweight tracks.",
        "check": "Verify the final canonical track identifier is unique."
    },
    {
        "rule": "Convert impossible bounded audio-feature values to missing",
        "reason": "Audio features expected on a 0-1 scale cannot validly fall outside that range.",
        "check": "Verify retained non-missing values are between 0 and 1."
    },
    {
        "rule": "Convert impossible popularity values to missing",
        "reason": "Track popularity is expected to be between 0 and 100.",
        "check": "Verify retained non-missing popularity values are between 0 and 100."
    },
    {
        "rule": "Convert nonpositive duration or tempo values to missing",
        "reason": "A usable track cannot have nonpositive duration or tempo.",
        "check": "Verify remaining non-missing values are positive."
    },
    {
        "rule": "Do not automatically delete unusual but valid observations",
        "reason": "Unusual tracks may be legitimate and potentially important to promotion analysis.",
        "check": "Inspect descriptive statistics and suspicious-value counts."
    },
    {
        "rule": "Require track identity and the primary performance metric for the final analysis",
        "reason": "Tracks must be identifiable and have track_popularity to support the project's primary comparison.",
        "check": "Verify no final records lack canonical_track_id or track_popularity."
    }
])

display(cleaning_decisions)

,rule,reason,check
0,Keep only project-relevant analytical columns,Focus the dataset on variables needed for trac...,Confirm retained columns match the documented ...
1,Preserve the original source and clean a copy,The source must remain reproducible and unchan...,Compare df_original dimensions before and afte...
2,Strip accidental leading/trailing whitespace f...,Whitespace can create false category differences.,Count values changed by whitespace removal.
3,Do not alter track IDs,Identifiers must preserve their original chara...,Compare IDs with the original data.
4,Convert release_date to a date type,Dates should have a consistent type for time-b...,Count failed date conversions.
5,Remove exact duplicate rows,Identical observations would double-count the ...,Verify no exact duplicates remain.
6,Use one record per track identifier,The analytical grain is one track and repeated...,Verify the final canonical track identifier is...
7,Convert impossible bounded audio-feature value...,Audio features expected on a 0-1 scale cannot ...,Verify retained non-missing values are between...
8,Convert impossible popularity values to missing,Track popularity is expected to be between 0 a...,Verify retained non-missing popularity values ...
9,Convert nonpositive duration or tempo values t...,A usable track cannot have nonpositive duratio...,Verify remaining non-missing values are positive.


In [7]:
missing_before = pd.DataFrame({
    "missing_before": df.isna().sum(),
    "percent_before": (df.isna().mean() * 100).round(2)
})

display(
    missing_before[missing_before["missing_before"] > 0]
    .sort_values("missing_before", ascending=False)
)

,missing_before,percent_before
streams_source,1172754,99.19
release_date,1159764,98.09
time_signature,22590,1.91
track_artist,19,0.00
title,5,0.00


In [8]:
text_columns = [
    "title",
    "track_artist",
    "genre_l1",
    "genre_l2",
    "streams_source"
]

text_changes = {}

for col in text_columns:
    before = df[col].copy()

    mask = df[col].notna()
    df.loc[mask, col] = (
        df.loc[mask, col]
        .astype(str)
        .str.strip()
        .replace("", np.nan)
    )

    changed = (
        before.fillna("<MISSING>").astype(str)
        != df[col].fillna("<MISSING>").astype(str)
    ).sum()

    text_changes[col] = changed

pd.DataFrame.from_dict(
    text_changes,
    orient="index",
    columns=["values_changed"]
)

,values_changed
title,0
track_artist,30
genre_l1,0
genre_l2,0
streams_source,0


In [9]:
release_date_original = df["release_date"].copy()

df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

failed_release_dates = (
    release_date_original.notna()
    & df["release_date"].isna()
).sum()

print("Failed release_date conversions:",
      f"{failed_release_dates:,}")

print("\nCurrent data types:")
print(df.dtypes.to_string())

Failed release_date conversions: 0

Current data types:
track_id                              str
spotify_id                            str
title                                 str
track_artist                          str
release_year                        int64
release_date               datetime64[us]
genre_l1                              str
genre_l2                              str
duration_ms                         int64
explicit_flag                        bool
energy                            float64
valence                           float64
danceability                      float64
acousticness                      float64
instrumentalness                  float64
speechiness                       float64
liveness                          float64
loudness                          float64
tempo                             float64
key                                 int64
mode                                int64
time_signature                    float64
track_popularity    

In [10]:
exact_duplicates_before = df.duplicated().sum()

print("Exact duplicate rows:",
      f"{exact_duplicates_before:,}")

print("\nRepeated spotify_id values:",
      f"{df.loc[df['spotify_id'].notna(), 'spotify_id'].duplicated().sum():,}")

print("Repeated track_id values:",
      f"{df.loc[df['track_id'].notna(), 'track_id'].duplicated().sum():,}")

Exact duplicate rows: 0

Repeated spotify_id values: 0
Repeated track_id values: 0


In [11]:
duplicate_spotify_examples = (
    df[df["spotify_id"].notna()]
    .loc[
        lambda x: x.duplicated(
            subset="spotify_id",
            keep=False
        )
    ]
    .sort_values("spotify_id")
)

display(duplicate_spotify_examples.head(20))

,track_id,spotify_id,title,track_artist,release_year,release_date,genre_l1,genre_l2,duration_ms,explicit_flag,...,liveness,loudness,tempo,key,mode,time_signature,track_popularity,streams_total,streams_source,streams_total_estimated


In [12]:
rows_before_exact_dedup = len(df)

df = df.drop_duplicates().copy()

exact_duplicates_removed = (
    rows_before_exact_dedup - len(df)
)

print("Exact duplicates removed:",
      f"{exact_duplicates_removed:,}")

Exact duplicates removed: 0


In [13]:
df["canonical_track_id"] = (
    df["spotify_id"]
    .fillna(df["track_id"])
)

print("Missing canonical track IDs:",
      f"{df['canonical_track_id'].isna().sum():,}")

print("Repeated canonical track IDs before deduplication:",
      f"{df.loc[df['canonical_track_id'].notna(), 'canonical_track_id'].duplicated().sum():,}")

Missing canonical track IDs: 0
Repeated canonical track IDs before deduplication: 0


In [14]:
rows_before_id_dedup = len(df)

df["_source_order"] = np.arange(len(df))

columns_for_completeness = [
    col for col in project_columns
    if col not in ["track_id", "spotify_id"]
]

df["_completeness"] = (
    df[columns_for_completeness]
    .notna()
    .sum(axis=1)
)

df = (
    df
    .sort_values(
        ["canonical_track_id", "_completeness", "_source_order"],
        ascending=[True, False, True],
        na_position="last"
    )
    .drop_duplicates(
        subset="canonical_track_id",
        keep="first"
    )
    .sort_values("_source_order")
    .copy()
)

id_duplicates_removed = (
    rows_before_id_dedup - len(df)
)

print("Repeated track records removed:",
      f"{id_duplicates_removed:,}")

print(
    "Repeated canonical IDs remaining:",
    f"{df.loc[df['canonical_track_id'].notna(), 'canonical_track_id'].duplicated().sum():,}"
)

Repeated track records removed: 0
Repeated canonical IDs remaining: 0


In [15]:
bounded_audio_features = [
    "energy",
    "valence",
    "danceability",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness"
]

suspicious_counts = {}

for col in bounded_audio_features:
    suspicious_counts[col] = (
        df[col].notna()
        & ~df[col].between(0, 1)
    ).sum()

suspicious_counts["track_popularity"] = (
    df["track_popularity"].notna()
    & ~df["track_popularity"].between(0, 100)
).sum()

suspicious_counts["duration_ms_nonpositive"] = (
    df["duration_ms"].notna()
    & (df["duration_ms"] <= 0)
).sum()

suspicious_counts["tempo_nonpositive"] = (
    df["tempo"].notna()
    & (df["tempo"] <= 0)
).sum()

suspicious_counts["streams_negative"] = (
    df["streams_total"].notna()
    & (df["streams_total"] < 0)
).sum()

suspicious_counts["release_year_outside_1900_2026"] = (
    df["release_year"].notna()
    & ~df["release_year"].between(1900, 2026)
).sum()

suspicious_table = pd.DataFrame.from_dict(
    suspicious_counts,
    orient="index",
    columns=["suspicious_count"]
)

display(suspicious_table)

,suspicious_count
energy,0
valence,0
danceability,0
acousticness,0
instrumentalness,0
speechiness,0
liveness,0
track_popularity,0
duration_ms_nonpositive,0
tempo_nonpositive,0


In [16]:
# Bounded Spotify audio features
for col in bounded_audio_features:
    invalid = df[col].notna() & ~df[col].between(0, 1)
    df.loc[invalid, col] = np.nan

# Popularity must be 0-100
invalid_popularity = (
    df["track_popularity"].notna()
    & ~df["track_popularity"].between(0, 100)
)
df.loc[invalid_popularity, "track_popularity"] = np.nan

# Duration must be positive
invalid_duration = (
    df["duration_ms"].notna()
    & (df["duration_ms"] <= 0)
)
df.loc[invalid_duration, "duration_ms"] = np.nan

# Tempo must be positive
invalid_tempo = (
    df["tempo"].notna()
    & (df["tempo"] <= 0)
)
df.loc[invalid_tempo, "tempo"] = np.nan

# Streams cannot be negative
invalid_streams = (
    df["streams_total"].notna()
    & (df["streams_total"] < 0)
)
df.loc[invalid_streams, "streams_total"] = np.nan

# Release year outside a defensible historical/current range
invalid_year = (
    df["release_year"].notna()
    & ~df["release_year"].between(1900, 2026)
)
df.loc[invalid_year, "release_year"] = np.nan

In [17]:
validation_results = {}

for col in bounded_audio_features:
    validation_results[col] = (
        df[col].dropna().between(0, 1).all()
    )

validation_results["track_popularity"] = (
    df["track_popularity"]
    .dropna()
    .between(0, 100)
    .all()
)

validation_results["duration_ms"] = (
    (df["duration_ms"].dropna() > 0).all()
)

validation_results["tempo"] = (
    (df["tempo"].dropna() > 0).all()
)

validation_results["streams_total"] = (
    (df["streams_total"].dropna() >= 0).all()
)

pd.DataFrame.from_dict(
    validation_results,
    orient="index",
    columns=["passes_validation"]
)

,passes_validation
energy,True
valence,True
danceability,True
acousticness,True
instrumentalness,True
speechiness,True
liveness,True
track_popularity,True
duration_ms,True
tempo,True


In [18]:
rows_before_required_fields = len(df)

missing_id_before_drop = df["canonical_track_id"].isna().sum()
missing_primary_metric_before_drop = df["track_popularity"].isna().sum()

df = df.dropna(
    subset=["canonical_track_id", "track_popularity"]
).copy()

required_field_rows_removed = (
    rows_before_required_fields - len(df)
)

print("Rows missing track identity:",
      f"{missing_id_before_drop:,}")

print("Rows missing primary metric:",
      f"{missing_primary_metric_before_drop:,}")

print("Rows removed for required fields:",
      f"{required_field_rows_removed:,}")

Rows missing track identity: 0
Rows missing primary metric: 0
Rows removed for required fields: 0


In [19]:
missing_after = pd.DataFrame({
    "missing_after": df.isna().sum(),
    "percent_after": (
        df.isna().mean() * 100
    ).round(2)
})

missing_comparison = (
    missing_before
    .join(missing_after, how="outer")
    .fillna(0)
)

display(
    missing_comparison[
        (missing_comparison["missing_before"] > 0)
        | (missing_comparison["missing_after"] > 0)
    ]
)

,missing_before,percent_before,missing_after,percent_after
release_date,1159764.0,98.09,1159764,98.09
streams_source,1172754.0,99.19,1172754,99.19
time_signature,22590.0,1.91,22590,1.91
title,5.0,0.00,5,0.00
track_artist,19.0,0.00,19,0.00


## Missing-Data Rule

Missing values are not automatically replaced with zero because zero has substantive meaning for many Spotify variables.

Tracks missing secondary audio characteristics or streaming information are retained when they can still contribute to analyses that do not require those fields.

Records without a usable track identifier or the primary performance measure (`track_popularity`) are excluded from the final analytical dataset because they cannot support the project's primary track-level comparison.

Analyses involving secondary variables will use the non-missing record count for the relevant metric.

In [20]:
print("Exact duplicates remaining:",
      f"{df.duplicated().sum():,}")

print(
    "Repeated canonical track IDs remaining:",
    f"{df['canonical_track_id'].duplicated().sum():,}"
)

print(
    "Missing canonical track IDs:",
    f"{df['canonical_track_id'].isna().sum():,}"
)

Exact duplicates remaining: 0
Repeated canonical track IDs remaining: 0
Missing canonical track IDs: 0


In [21]:
reconciliation = pd.DataFrame({
    "step": [
        "Original source rows",
        "Exact duplicate rows removed",
        "Repeated track records removed",
        "Rows removed for missing required fields",
        "Final analytical rows"
    ],
    "count": [
        len(df_original),
        exact_duplicates_removed,
        id_duplicates_removed,
        required_field_rows_removed,
        len(df)
    ]
})

display(reconciliation)

accounted_final = (
    len(df_original)
    - exact_duplicates_removed
    - id_duplicates_removed
    - required_field_rows_removed
)

print("Calculated final rows:", f"{accounted_final:,}")
print("Actual final rows:", f"{len(df):,}")
print("Counts reconcile:", accounted_final == len(df))

,step,count
0,Original source rows,1182354
1,Exact duplicate rows removed,0
2,Repeated track records removed,0
3,Rows removed for missing required fields,0
4,Final analytical rows,1182354


Calculated final rows: 1,182,354
Actual final rows: 1,182,354
Counts reconcile: True


In [22]:
metric_summary = pd.DataFrame({
    "metric": [
        "track_popularity",
        "streams_total",
        "danceability",
        "energy",
        "valence"
    ]
})

metric_summary["non_missing_records"] = metric_summary["metric"].apply(
    lambda col: df[col].notna().sum()
)

metric_summary["missing_records"] = metric_summary["metric"].apply(
    lambda col: df[col].isna().sum()
)

metric_summary["mean"] = metric_summary["metric"].apply(
    lambda col: df[col].mean()
)

display(metric_summary)

,metric,non_missing_records,missing_records,mean
0,track_popularity,1182354,0,1.869929e+01
1,streams_total,1182354,0,4.664846e+06
2,danceability,1182354,0,5.396589e-01
3,energy,1182354,0,6.407276e-01
4,valence,1182354,0,4.566848e-01


In [23]:
genre_summary = (
    df
    .dropna(subset=["genre_l1"])
    .groupby("genre_l1", as_index=False)
    .agg(
        tracks=("canonical_track_id", "count"),
        avg_popularity=("track_popularity", "mean"),
        tracks_with_streams=("streams_total", "count"),
        avg_streams=("streams_total", "mean")
    )
    .sort_values("tracks", ascending=False)
)

display(genre_summary.head(20))

,genre_l1,tracks,avg_popularity,tracks_with_streams,avg_streams
3,Electronic,307754,17.133152,307754,3.531977e+06
14,Rock,135807,21.618576,135807,4.999363e+06
11,Pop,131828,18.388666,131828,5.679040e+06
10,Metal,98541,15.164784,98541,3.700075e+06
9,Latin,93274,14.785042,93274,5.160245e+06
17,World,79530,17.691588,79530,4.711568e+06
1,Classical,58223,16.254161,58223,4.810709e+06
4,Folk,51302,24.280418,51302,4.858167e+06
12,R&B,41155,23.017325,41155,5.403232e+06
13,Reggae,37573,15.412637,37573,7.254551e+06


In [24]:
# Select five records to verify different cleaning decisions
record_check_candidates = {}

# 1. Artist name with leading/trailing whitespace
mask = (
    df_original["track_artist"].notna()
    & (
        df_original["track_artist"].astype(str)
        != df_original["track_artist"].astype(str).str.strip()
    )
)

if mask.any():
    record_check_candidates["text_whitespace"] = (
        "track_artist",
        df_original.loc[mask].iloc[0]
    )

# 2. Missing release date — record should still be retained
mask = df_original["release_date"].isna()

if mask.any():
    record_check_candidates["missing_release_date"] = (
        "release_date",
        df_original.loc[mask].iloc[0]
    )

# 3. Missing time signature — record should still be retained
mask = df_original["time_signature"].isna()

if mask.any():
    record_check_candidates["missing_time_signature"] = (
        "time_signature",
        df_original.loc[mask].iloc[0]
    )

# 4. Track whose stream total is marked as estimated
mask = df_original["streams_total_estimated"].eq(True)

if mask.any():
    record_check_candidates["estimated_streams"] = (
        "streams_total_estimated",
        df_original.loc[mask].iloc[0]
    )

# 5. Normal valid record — should pass through unchanged
normal_mask = (
    df_original["spotify_id"].notna()
    & df_original["track_popularity"].between(0, 100)
    & df_original["energy"].between(0, 1)
    & df_original["danceability"].between(0, 1)
)

if normal_mask.any():
    record_check_candidates["normal_record"] = (
        "spotify_id",
        df_original.loc[normal_mask].iloc[0]
    )

print("Record-check cases found:")
for case, (column, row) in record_check_candidates.items():
    print(case, "->", column)

Record-check cases found:
text_whitespace -> track_artist
missing_release_date -> release_date
missing_time_signature -> time_signature
estimated_streams -> streams_total_estimated
normal_record -> spotify_id


In [25]:
check_rows = []

for case, (column, row) in record_check_candidates.items():

    original_id = (
        row["spotify_id"]
        if pd.notna(row["spotify_id"])
        else row["track_id"]
    )

    actual = df[
        df["canonical_track_id"] == original_id
    ]

    # Define expected result and reason BEFORE comparing with actual result
    if case == "text_whitespace":
        expected = str(row[column]).strip()
        reason = "Leading/trailing whitespace should be removed from artist names."

    elif case == "missing_release_date":
        expected = np.nan
        reason = "A missing release date should remain missing and should not cause the track to be deleted."

    elif case == "missing_time_signature":
        expected = np.nan
        reason = "Missing optional audio metadata should remain missing rather than being imputed or causing row deletion."

    elif case == "estimated_streams":
        expected = row[column]
        reason = "The estimated-stream flag should be preserved so estimated totals remain identifiable."

    elif case == "normal_record":
        expected = row[column]
        reason = "A valid record should pass through cleaning without changing its Spotify ID."

    actual_value = (
        actual.iloc[0][column]
        if not actual.empty
        else np.nan
    )

    # Determine whether actual result matches expected result
    if pd.isna(expected) and pd.isna(actual_value):
        passed = not actual.empty
    else:
        passed = (
            not actual.empty
            and actual_value == expected
        )

    check_rows.append({
        "case": case,
        "track_id_checked": original_id,
        "column_checked": column,
        "original_value": row[column],
        "expected_result": expected,
        "reason": reason,
        "actual_value": actual_value,
        "record_present_after_cleaning": not actual.empty,
        "pass": passed
    })

record_checks = pd.DataFrame(check_rows)

display(record_checks)
print(
    "All five record checks pass:",
    len(record_checks) == 5 and record_checks["pass"].all()
)

,case,track_id_checked,column_checked,original_value,expected_result,reason,actual_value,record_present_after_cleaning,pass
0,text_whitespace,5cXq69EbEhTaWE7lUTOwsK,track_artist,The Wanderer,The Wanderer,Leading/trailing whitespace should be removed ...,The Wanderer,True,True
1,missing_release_date,0yLdNVWF3Srea0uzk55zFn,release_date,NaN,NaN,A missing release date should remain missing a...,NaT,True,True
2,missing_time_signature,0sf12qNH5qcw8qpgymFOqD,time_signature,NaN,NaN,Missing optional audio metadata should remain ...,NaN,True,True
3,estimated_streams,4nrPB8O7Y7wsOCJdgXkthe,streams_total_estimated,True,True,The estimated-stream flag should be preserved ...,True,True,True
4,normal_record,0yLdNVWF3Srea0uzk55zFn,spotify_id,0yLdNVWF3Srea0uzk55zFn,0yLdNVWF3Srea0uzk55zFn,A valid record should pass through cleaning wi...,0yLdNVWF3Srea0uzk55zFn,True,True


All five record checks pass: True


## Merge Verification

**Status: Not applicable**

This cleaning workflow uses a single source dataset (`spotify_full.csv`) and does not merge or join separate tables. Therefore, there are no merge keys, unmatched join keys, or merge-generated rows to verify.

In [26]:
# Remove temporary helper columns before export
df_clean = df.drop(
    columns=["_source_order", "_completeness"],
    errors="ignore"
).copy()

print("=== FINAL CLEANED DATASET ===")
print("Rows:", f"{len(df_clean):,}")
print("Columns:", f"{df_clean.shape[1]:,}")

print("\nExact duplicates:",
      f"{df_clean.duplicated().sum():,}")

print("Duplicate track IDs:",
      f"{df_clean['canonical_track_id'].duplicated().sum():,}")

print("Missing primary metric:",
      f"{df_clean['track_popularity'].isna().sum():,}")

display(df_clean.head())

=== FINAL CLEANED DATASET ===
Rows: 1,182,354
Columns: 27

Exact duplicates: 0
Duplicate track IDs: 0
Missing primary metric: 0


,track_id,spotify_id,title,track_artist,release_year,release_date,genre_l1,genre_l2,duration_ms,explicit_flag,...,loudness,tempo,key,mode,time_signature,track_popularity,streams_total,streams_source,streams_total_estimated,canonical_track_id
0,spotify:track:0yLdNVWF3Srea0uzk55zFn,0yLdNVWF3Srea0uzk55zFn,Flowers,Miley Cyrus,2023.0,NaT,Pop,Pop,200455.0,False,...,-4.325,117.999,0,1,4.0,100.0,1.316856e+09,arc2_2023_top,False,0yLdNVWF3Srea0uzk55zFn
1,spotify:track:0sf12qNH5qcw8qpgymFOqD,0sf12qNH5qcw8qpgymFOqD,Blinding Lights,The Weeknd,2019.0,2019-11-29,Pop,Dance Pop,201573.0,False,...,-4.075,171.017,1,1,NaN,98.0,5.859040e+05,arc7_chart_dump,False,0sf12qNH5qcw8qpgymFOqD
2,spotify:track:21jGcNKet2qwijlDFuPiPb,21jGcNKet2qwijlDFuPiPb,Circles,Post Malone,2019.0,2019-09-06,Latin,Latin Hip-Hop,215280.0,False,...,-3.497,120.042,0,1,NaN,98.0,5.985218e+08,arc7_chart_dump,False,21jGcNKet2qwijlDFuPiPb
3,spotify:track:2b8fOow8UzyDFAE27YhOZM,2b8fOow8UzyDFAE27YhOZM,Memories,Maroon 5,2019.0,2019-09-20,Pop,Dance Pop,189486.0,False,...,-7.209,91.019,11,1,NaN,98.0,1.596861e+08,arc7_chart_dump,False,2b8fOow8UzyDFAE27YhOZM
4,spotify:track:7k4t7uLgtOxPwTpFmtJNTY,7k4t7uLgtOxPwTpFmtJNTY,Tusa,KAROL G,2019.0,2019-11-07,Pop,Dance Pop,200960.0,True,...,-3.280,101.085,2,1,NaN,98.0,5.682441e+07,arc7_chart_dump,False,7k4t7uLgtOxPwTpFmtJNTY


In [27]:
difficult_ids = (
    record_checks["track_id_checked"]
    .dropna()
    .unique()
    .tolist()
)

difficult_sample = df_clean[
    df_clean["canonical_track_id"].isin(difficult_ids)
].copy()

remaining_needed = max(
    0,
    50 - len(difficult_sample)
)

random_sample = (
    df_clean[
        ~df_clean["canonical_track_id"].isin(difficult_ids)
    ]
    .sample(
        n=min(remaining_needed, len(df_clean)),
        random_state=576
    )
)

github_sample = (
    pd.concat(
        [difficult_sample, random_sample],
        ignore_index=True
    )
    .head(50)
)

print("GitHub sample rows:", len(github_sample))

display(github_sample.head())

GitHub sample rows: 50


,track_id,spotify_id,title,track_artist,release_year,release_date,genre_l1,genre_l2,duration_ms,explicit_flag,...,loudness,tempo,key,mode,time_signature,track_popularity,streams_total,streams_source,streams_total_estimated,canonical_track_id
0,spotify:track:0yLdNVWF3Srea0uzk55zFn,0yLdNVWF3Srea0uzk55zFn,Flowers,Miley Cyrus,2023.0,NaT,Pop,Pop,200455.0,False,...,-4.325,117.999,0,1,4.0,100.0,1.316856e+09,arc2_2023_top,False,0yLdNVWF3Srea0uzk55zFn
1,spotify:track:0sf12qNH5qcw8qpgymFOqD,0sf12qNH5qcw8qpgymFOqD,Blinding Lights,The Weeknd,2019.0,2019-11-29,Pop,Dance Pop,201573.0,False,...,-4.075,171.017,1,1,NaN,98.0,5.859040e+05,arc7_chart_dump,False,0sf12qNH5qcw8qpgymFOqD
2,spotify:track:4nrPB8O7Y7wsOCJdgXkthe,4nrPB8O7Y7wsOCJdgXkthe,"Shakira: Bzrp Music Sessions, Vol. 53",Bizarrap,2023.0,NaT,Hip-Hop,Hip-Hop,218289.0,False,...,-5.600,122.104,2,0,4.0,96.0,1.682377e+08,NaN,True,4nrPB8O7Y7wsOCJdgXkthe
3,spotify:track:5cXq69EbEhTaWE7lUTOwsK,5cXq69EbEhTaWE7lUTOwsK,We're All Going Home,The Wanderer,2019.0,NaT,Folk,Singer-Songwriter,184746.0,False,...,-11.835,167.825,2,1,4.0,53.0,1.633113e+06,NaN,True,5cXq69EbEhTaWE7lUTOwsK
4,spotify:track:4sccZEyUgxvukqfpRujxt7,4sccZEyUgxvukqfpRujxt7,Жду весну,Vladimir Troshin,2008.0,NaT,Pop,Romantic,232093.0,False,...,-17.072,89.681,8,1,4.0,0.0,3.637466e+06,NaN,True,4sccZEyUgxvukqfpRujxt7


## Submitted Data Sample

The submitted sample contains up to 50 cleaned rows. Difficult records used during verification are included when available, and the remaining rows are selected using a reproducible random sample (`random_state=576`).

This provides examples of both difficult cleaning cases and typical records without uploading the full dataset.

In [28]:
github_sample.to_csv(
    "spotify_cleaned_sample.csv",
    index=False
)

# Save the full cleaned analytical dataset locally.
# Add this filename to .gitignore before pushing.
df_clean.to_csv(
    "spotify_cleaned_full.csv",
    index=False
)

print("Created:")
print("- spotify_cleaned_sample.csv")
print("- spotify_cleaned_full.csv")

Created:
- spotify_cleaned_sample.csv
- spotify_cleaned_full.csv


In [29]:
with open("requirements.txt", "w") as f:
    f.write(f"pandas=={pd.__version__}\n")
    f.write(f"numpy=={np.__version__}\n")

print("requirements.txt created")

requirements.txt created


In [30]:
final_checks = {
    "original_dataframe_still_exists": df_original is not None,
    "cleaned_dataset_not_empty": len(df_clean) > 0,
    "canonical_id_complete": df_clean["canonical_track_id"].notna().all(),
    "canonical_id_unique": df_clean["canonical_track_id"].is_unique,
    "no_exact_duplicates": not df_clean.duplicated().any(),
    "primary_metric_complete": df_clean["track_popularity"].notna().all(),
    "popularity_valid": df_clean["track_popularity"].between(0, 100).all(),
    "energy_valid": df_clean["energy"].dropna().between(0, 1).all(),
    "danceability_valid": df_clean["danceability"].dropna().between(0, 1).all(),
    "valence_valid": df_clean["valence"].dropna().between(0, 1).all(),
    "counts_reconcile": accounted_final == len(df_clean),
    "github_sample_at_most_50": len(github_sample) <= 50
}

final_checks = pd.Series(
    final_checks,
    name="passes"
)

display(final_checks)

print(
    "\nALL FINAL CHECKS PASS:",
    final_checks.all()
)

original_dataframe_still_exists    True
cleaned_dataset_not_empty          True
canonical_id_complete              True
canonical_id_unique                True
no_exact_duplicates                True
primary_metric_complete            True
popularity_valid                   True
energy_valid                       True
danceability_valid                 True
valence_valid                      True
counts_reconcile                   True
github_sample_at_most_50           True
Name: passes, dtype: bool


ALL FINAL CHECKS PASS: True


## Reproducibility

To reproduce the cleaning process:

1. Place the original `spotify_full.csv` file in the project directory.
2. Install the packages listed in `requirements.txt`.
3. Open `cleaning.ipynb`.
4. Restart the Jupyter kernel.
5. Run all cells from top to bottom.
6. Confirm that all final validation checks pass.
7. Confirm that `spotify_cleaned_sample.csv` and `spotify_cleaned_full.csv` are recreated.

The original `spotify_full.csv` is never modified by the notebook.

## Remaining Limitations

1. Historical association does not establish that a track characteristic causes higher popularity.
2. Track popularity may reflect factors outside the available dataset, including marketing spend, artist recognition, playlist placement, social-media activity, and label support.
3. Streaming information is missing for some records and some streaming values may be estimated, so streaming comparisons use only available observations and should be interpreted cautiously.
4. Genre classifications may be broad or incomplete for some tracks.
5. Missing audio characteristics are retained when possible rather than replaced with invented values, so analyses involving those variables may use fewer observations.
6. Duplicate resolution selects the most complete record for a track identifier. When equally complete records disagree, the original source order is preserved rather than selecting the record with the strongest outcome.
7. The analysis can support promotion prioritization but cannot guarantee that a promoted track will become successful.

In [31]:
print(df_clean.shape)
print(len(df_clean))

(1182354, 27)
1182354
